# Day 48：里程碑项目 3（Part 1）—— 多 Agent 智能办公助手核心编排与工具集

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 48）！今天我们将实战演示 **多 Agent 智能办公助手（`v0.4.0-day48`）**：由 `OfficeSupervisor` 统筹 `ResearchAgent`、`CalendarAgent`、`MailAgent` 三大专职 Agent，集成 6 大企业办公工具，并通过 `SQLiteCheckpointer` 与 `interrupt_before` 实现高危操作（`create_meeting`、`send_email`）的人工审批挂起与跨实例断点恢复。

## 1. 环境初始化与导入 `project_iterations/day_48` 模块
定位仓库根目录并将 `project_iterations/day_48` 加入 `sys.path`。

In [1]:
import json
import sys
import tempfile
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "project_iterations" / "day_48").exists()), None)
assert repo_root is not None, "未找到 project_iterations/day_48 目录"

day48_dir = repo_root / "project_iterations" / "day_48"
if str(day48_dir) not in sys.path:
    sys.path.insert(0, str(day48_dir))

from zhishu_agent import (
    OfficeSupervisor,
    OfficeWorkspace,
    SQLiteCheckpointer,
    TaskStatus,
    create_default_tools,
)

print(f"已加载模块路径: {day48_dir}")

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_48


## 2. 6 大企业办公工具验证
演示 `list_calendar`、`create_meeting`、`draft_email`、`send_email`、`search_company_docs` 与 `web_search`。

In [2]:
ws = OfficeWorkspace()
tools = create_default_tools(ws)
print("已注册办公工具:", list(tools.keys()))

cal = tools["list_calendar"].invoke(date="2026-11-16")
docs = tools["search_company_docs"].invoke(query="Q4 AI 战略", top_k=2)
web = tools["web_search"].invoke(query="SQLite 断点恢复", top_k=1)

print("当日日程数量:", cal["count"])
print("内部文档命中:", [d["title"] for d in docs["results"]])
print("联网检索命中:", [w["title"] for w in web["results"]])

已注册办公工具: ['list_calendar', 'create_meeting', 'draft_email', 'send_email', 'search_company_docs', 'web_search']
当日日程数量: 2
内部文档命中: ['星辰数科 Q4 AI 战略与预算规划', '智枢平台 v0.4.0 多 Agent 办公助手架构规范']
联网检索命中: ['基于 SQLite 的长程智能体断点恢复设计']


## 3. 多 Agent 协同编排与 SQLite 跨实例断点恢复（HITL）
使用 `tempfile.TemporaryDirectory()` 创建临时 SQLite 数据库，演示：
1. `OfficeSupervisor.run()` 在 `create_meeting` 前自动中断挂起；
2. **新建另一个 `OfficeSupervisor` 实例**（模拟服务重启）加载同一 SQLite 文件，批准并微调会议时间；
3. 推进到 `MailAgent` 在 `send_email` 前再次挂起，人工批准后完成全流程。

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / "office_hitl.db"
    workspace = OfficeWorkspace()

    sup_before_restart = OfficeSupervisor(
        workspace=workspace,
        checkpointer=SQLiteCheckpointer(db_path),
        interrupt_before={"create_meeting", "send_email"},
    )

    step1 = sup_before_restart.run(
        "请调研 Q4 战略文档，预约会议并发送邮件通知张工",
        thread_id="nb-thread-001",
    )
    print("[阶段 1] 状态:", step1["status"], "| 挂起工具:", step1["pending_action"]["tool_name"])
    assert step1["status"] == TaskStatus.INTERRUPTED.value

    # 模拟服务重启，创建新的 Supervisor 实例读取同一 SQLite
    sup_after_restart = OfficeSupervisor(
        workspace=workspace,
        checkpointer=SQLiteCheckpointer(db_path),
        interrupt_before={"create_meeting", "send_email"},
    )
    step2 = sup_after_restart.resume(
        "nb-thread-001",
        approved=True,
        modified_args={"start_time": "16:30"},
        feedback="微调至 16:30 避开冲突",
    )
    print("[阶段 2] 状态:", step2["status"], "| 下一个挂起工具:", step2["pending_action"]["tool_name"])
    assert step2["pending_action"]["tool_name"] == "send_email"

    step3 = sup_after_restart.resume("nb-thread-001", approved=True, feedback="确认发送")
    print("[阶段 3] 最终状态:", step3["status"])
    print(step3["final_response"])
    print("SQLite 检查点历史数:", len(sup_after_restart.get_history("nb-thread-001")))

[阶段 1] 状态: interrupted | 挂起工具: create_meeting
[阶段 2] 状态: interrupted | 下一个挂起工具: send_email
[阶段 3] 最终状态: completed
已完成办公协同请求：请调研 Q4 战略文档，预约会议并发送邮件通知张工
- [ResearchAgent] 已检索内部文档 2 篇（智枢平台 v0.4.0 多 Agent 办公助手架构规范, 企业差旅与会议审批制度（2026 修订版））与外部网页 2 篇（Multi-Agent Supervisor 与 Human-in-the-Loop 最佳实践, 2026 企业智能办公助手行业基准报告）。
- [CalendarAgent] 已创建会议 MTG-003: 智枢多 Agent 项目对齐会 (2026-11-16 16:30)
- [MailAgent] 已发送邮件 MSG-001 至 linxiao@zhishu.cn, zhanggong@zhishu.cn
SQLite 检查点历史数: 7


## 4. 高危操作人工拒绝（`approved=False`）验证
验证审批拒绝后任务安全进入 `rejected` 状态，且不会真正发送邮件。

In [4]:
ws_rej = OfficeWorkspace()
sup_rej = OfficeSupervisor(
    workspace=ws_rej,
    checkpointer=SQLiteCheckpointer(":memory:"),
    interrupt_before={"send_email"},
)

r_start = sup_rej.run("请发送邮件通知全员", thread_id="nb-reject-01")
assert r_start["status"] == TaskStatus.INTERRUPTED.value

r_end = sup_rej.resume("nb-reject-01", approved=False, feedback="收件人范围需缩小至研发部")
print("拒绝后状态:", r_end["status"])
print("拒绝说明:", r_end["final_response"])
assert r_end["status"] == TaskStatus.REJECTED.value
assert len(ws_rej.sent_emails) == 0

拒绝后状态: rejected
拒绝说明: 操作 send_email 已被人工拒绝：收件人范围需缩小至研发部
